# Kineret: final experiment workflow

This notebook is the single activation point for the research package. The primary task is **observe 96 hours, predict outcomes in (96, 336] hours**. Only K-sensitivity changes the observation window. Training augmentation remains K=1..13.

The benchmark includes the original seven arms plus one SAX arm (global training normalization, four-hour means, seven symbols, no imputation or QA).

The agreed study period starts **2022-08-01** (thesis evaluation methods and appendix; JAMIA methods). Existing checkpoints keep their original training provenance, including July-start training. New evaluations filter admissions to August onward without changing the original person split or outcome heads. They are not a claim that old models were trained on an August-only cohort.

All new outputs go to a separate experiment directory. Frozen thesis artifacts live in `results/full_bench_results/`, including the exact executed notebook and configuration. This notebook does not rebuild the old cache, overwrite checkpoints, or continue training during inference.


In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'kineret').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault('INTERVENE_PROGRESS', 'tqdm')
from kineret.config import paths, data_config as C
from kineret.benchmark import ARM_ORDER
from kineret.experiments import workflow as W
from kineret.experiments.figures import export_figures, window_figure
from kineret.experiments.inference import checkpoint_for, recover_strats_preprocessor

# All activation choices and resource limits live here.
SOURCE_OUTPUTS = ROOT / 'outputs'             # original per-arm predictions / STraTS weights
CHECKPOINT_ROOT = ROOT / 'checkpoints'        # original INTERVenE checkpoints
SOURCE_COHORT = Path(paths.COHORT_PKL)         # original split manifest: do not rebuild it
EXPERIMENT_ROOT = ROOT / 'outputs' / 'final_recipe_aug2022'
START, END, PRIMARY_K = '2022-08-01', '2025-12-31', 4
BOOTSTRAP, SEED, DEVICE = 2000, 2023, None     # None: CUDA if available
ARMS = list(ARM_ORDER)
RUN = dict(train_missing=True, primary=True, subgroups=True, calibration=True, paired=True,
           multi_k=False, loo=False, umap=False, xai=False, stability=False)
K_VALUES = [2, 3, 4, 5, 6]
# Include QA to test its effect across windows. Missing legacy LR bundles are reported, never retrained.
K_ARMS = ['strats', 'strats_qa', 'intervene_std', 'intervene_kb', 'intervene_kb_qa']
K_ARMS.append('intervene_sax')
# QA and the raw baseline are central; add intervene_std for a 4-arm / 24-fit LOO.
LOO_ARMS = ['strats', 'intervene_kb', 'intervene_kb_qa']
LOO_HOSPITALS = None                         # None: all sites; a list allows staged execution
VISITS_MASTER = Path(paths.VISITS_MASTER_FILE)
DEMOGRAPHIC_COLUMNS = {'age': 'age', 'sex': 'sex'}  # canonical name -> actual context column

# Exact deployed Mediator checkout + Kineret repository pickle are needed only for stability.
MEDIATOR_ROOT = ROOT.parent / 'Mediator'
MEDIATOR_REPOSITORY = None                  # REQUIRED: exact deployed Kineret repository pickle
STABILITY_PEOPLE, STABILITY_REPEATS = 500, 2
XAI_ARM, XAI_FIT_PEOPLE, XAI_VALIDATION_PEOPLE, XAI_TEST_PEOPLE = 'intervene_kb_qa', 300, 64, 300
XAI_EPOCHS, XAI_TOKENS_PER_ADMISSION = 20, 256
UMAP_ARM = 'intervene_kb_qa'
ARM_SOURCES = {}
SCHEMA_PATHS = {}                            # optional per-arm recovered STraTS schemas
STD_TABLE = Path(paths.PROCESSED_DIR) / 'std_temporal_data.csv'
STD_REPO = Path(paths.PROCESSED_DIR) / 'tak_repo_std.json'

from kineret.sax import RECIPE as SAX_RECIPE
study = json.loads((ROOT / 'kineret/config/final_recipe.json').read_text())
study.update(date_range_start=START, date_range_end=END, eval_context_days=PRIMARY_K)
C.configure_study(**study)
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
W.write_json(EXPERIMENT_ROOT / 'requested_recipe.json', dict(study=study, run=RUN,
    arms=ARMS, sax_recipe=SAX_RECIPE, k_values=K_VALUES, k_arms=K_ARMS, loo_arms=LOO_ARMS,
    checkpoint_training_period='unchanged; see original run metadata'))
W.inventory(str(SOURCE_OUTPUTS), str(CHECKPOINT_ROOT))


## Load the original cohort, then apply the agreed evaluation dates

The original cached cohort is necessary to retain the trained model's split and heads. For a genuinely new deployment, build a new cohort with `build_cohort()` under this configuration and save it to a new path; do not replace a cohort belonging to existing models. The source cache can include July because it records the historical split. Loaded evaluation tables exclude those admissions.


In [ ]:
if SOURCE_COHORT.exists():
    cohort = W.load_study(SOURCE_COHORT, START, END)
else:
    available = W.inventory(str(SOURCE_OUTPUTS), str(CHECKPOINT_ROOT))
    if available.checkpoint_available.any() or available.test_predictions.any():
        raise FileNotFoundError('Existing models require their original cohort/split artifact; set SOURCE_COHORT to it.')
    from kineret.cohort import build_cohort
    cohort = build_cohort()  # new deployment only; uses the August-start configuration above
    SOURCE_COHORT = EXPERIMENT_ROOT / 'new_cohort.pkl'
    cohort.save(str(SOURCE_COHORT))
    cohort.meta['checkpoint_training_period_unchanged'] = False
from kineret.experiments.cohorts import export_split, people, hospital_series, support_table
export_split(cohort, EXPERIMENT_ROOT)
print(cohort.patients.groupby('split').size())
print('Retained outcome heads:', cohort.outcome_names)
print('Evaluation sample counts:', cohort.samples().groupby('split').size().to_dict())
display(support_table(cohort, PRIMARY_K))


## Legacy STraTS preprocessing recovery (only if its schema is missing)

Old STraTS weights did not store the vocabulary/scalers. The helper below replays the original seeded preprocessing from the **original untouched processed pickle**, without training. Save its result in the new experiment directory. Do not use an August-rebuilt or different-fold pickle. After recovery compare K=4 inference against the archived predictions before relying on multi-K results.

Old LogReg outputs have coefficients but no complete checkpoint/intercept/scalers. K=4 can reuse saved predictions; new inference requires a complete model bundle from a future explicitly launched fit. It is never silently retrained.


In [ ]:
RECOVER_STRATS_SCHEMAS = False
if RECOVER_STRATS_SCHEMAS:
    for arm, suffix in [('strats', 'noqa'), ('strats_qa', 'qa')]:
        destination = EXPERIMENT_ROOT / 'recovered_preprocessing' / f'{arm}.pkl'
        if not destination.exists():
            recover_strats_preprocessor(arm, str(SOURCE_OUTPUTS),
                Path(paths.PROCESSED_DIR) / f'kineret_{suffix}.pkl', destination)
        SCHEMA_PATHS[arm] = str(destination)


## Train missing primary arms

`ARMS` selects the benchmark models, including SAX. `RUN['train_missing']` enables training for missing arms uniformly. Existing checkpoints or saved test predictions prevent retraining; missing inference metadata is repaired or reported. New models use isolated directories and are passed to evaluation through `ARM_SOURCES`.


In [ ]:
if RUN['train_missing']:
    ARM_SOURCES = W.train_missing_main(cohort, EXPERIMENT_ROOT / 'new_primary_models', ARMS,
        source_outputs=str(SOURCE_OUTPUTS), checkpoint_root=str(CHECKPOINT_ROOT),
        device=DEVICE, n_resamples=BOOTSTRAP)

def selected_checkpoint(arm):
    source = ARM_SOURCES.get(arm, {})
    return checkpoint_for(arm, source.get('outputs', str(SOURCE_OUTPUTS)),
                          source.get('checkpoints', str(CHECKPOINT_ROOT)))


## Eight-arm primary evaluation: existing predictions or frozen checkpoints

Every new CI resamples **people with all their admissions**. Point estimates are calculated on the observed test set, not averages of bootstrap estimates. Degenerate outcome draws are counted and unreliable intervals flagged. Missing artifacts are exported in `status.csv`; a partial table must not be reported as the complete ladder.


In [ ]:
if RUN['primary']:
    predictions, metrics, status = W.evaluate(cohort, EXPERIMENT_ROOT / 'primary', arms=ARMS,
        k=PRIMARY_K, source_outputs=str(SOURCE_OUTPUTS), checkpoint_root=str(CHECKPOINT_ROOT),
        device=DEVICE, n_resamples=BOOTSTRAP, schema_paths=SCHEMA_PATHS, arm_sources=ARM_SOURCES,
        std_table=str(STD_TABLE), std_repo=str(STD_REPO))
    display(status)
    W.require_predictions(predictions, ARMS)
    display(metrics[(metrics['outcome'] == 'weighted') & (metrics['metric'] == 'auprc')]
            if not metrics.empty else metrics)


## Paired comparisons, demographics and calibration

Primary contrasts: KB vs sigma, KB vs SAX, SAX vs sigma, KB vs STraTS, and QA vs no-QA within each architecture. Report paired differences and CIs; overlapping separate model intervals do not establish a null effect. BH families are explicit in the output: outcomes within a contrast/metric, and headline contrasts within a metric/average. Macro results are secondary.

Subgroups are descriptive performance heterogeneity, not a complete fairness assessment. Report admissions, people, positives, prevalence and CIs; include missing demographics as a separate category. Temperature is fit only on disjoint validation people. Without validation predictions, report raw calibration only, never test-self-fit calibration. ECE uses 15 uniform bins; plots use 10 quantile bins with bin counts exported.


In [ ]:
demographics = cohort.context.reset_index()[['PatientId']].copy()
for canonical, actual in DEMOGRAPHIC_COLUMNS.items():
    if actual not in cohort.context:
        if RUN['subgroups']:
            raise KeyError(f'Map {canonical!r} to an available context column: {list(cohort.context.columns)}')
    else:
        demographics[canonical] = cohort.context[actual].to_numpy()
if VISITS_MASTER.exists():
    demographics['hospital'] = demographics.PatientId.map(hospital_series(cohort, VISITS_MASTER))
if RUN['primary'] and any(RUN[key] for key in ('subgroups', 'calibration', 'paired')):
    paired = W.analyze(predictions, EXPERIMENT_ROOT / 'primary',
        demographics=demographics if RUN['subgroups'] else None, n_resamples=BOOTSTRAP,
        do_paired=RUN['paired'], do_calibration=RUN['calibration'])
    display(paired)
    export_figures(EXPERIMENT_ROOT / 'primary')


## Multiple evaluation windows: frozen models, changing eligible admissions

At each K, include original test people whose admissions reach K; keep all eight original outcome heads. The eligible cohort and outcome prevalence change with K, as does the remaining horizon to day 14. This is a test of relative model performance across clinical prediction times, not a pure causal estimate of adding one day of history. No training or normalization fitting occurs.


In [ ]:
if RUN['multi_k']:
    from kineret.multi_k import sweep_arms
    window_metrics = sweep_arms(K_ARMS, K_VALUES, cohort, EXPERIMENT_ROOT / 'multi_k',
        source_outputs=str(SOURCE_OUTPUTS), checkpoint_root=str(CHECKPOINT_ROOT),
        device=DEVICE, n_resamples=BOOTSTRAP, schema_paths=SCHEMA_PATHS, arm_sources=ARM_SOURCES,
        std_table=str(STD_TABLE), std_repo=str(STD_REPO))
    window_figure(window_metrics, EXPERIMENT_ROOT / 'multi_k')
    display(window_metrics.query("outcome == 'weighted' and metric == 'auprc'"))


## Hospital leave-one-out: selected arms, sequential and resumable

Default: STraTS, KB, KB+QA (18 fits for six hospitals). Add sigma only if the knowledge-base contrast at unseen sites is needed. Each hospital stays in test; people with admissions there are excluded from other sites' training/validation. Hospital is split metadata, never accidentally added to the numeric context. Scalers, vocabularies and sigma cutoffs are fitted within each fold. The fixed target heads are retained; rare/unsupported targets are flagged rather than reselected using the held-out site.

Pooled models cannot be reused as LOO models. Completed **matching fold** checkpoints are reused; a changed fold manifest stops execution. Folds run one at a time, in separate checkpoint/cache/output directories. Interpret per-site AUPRC together with prevalence; these six sites do not establish generalization to every hospital.


In [ ]:
if RUN['loo']:
    loo = W.hospital_loo(cohort, EXPERIMENT_ROOT / 'loo', VISITS_MASTER,
        arms=LOO_ARMS, hospitals=LOO_HOSPITALS, device=DEVICE, n_resamples=BOOTSTRAP)
    display(loo.query("outcome == 'weighted' and metric == 'auprc'"))


## Trained embedding hierarchy

UMAP is descriptive, paired with original-space cosine distances. Fixed timestamp, disabled dropout, final trained embedder, no prediction-model training. It characterizes hierarchy in the trained representation, not the hierarchy's causal contribution to accuracy.


In [ ]:
if RUN['umap']:
    from kineret.experiments.inference import load_encoder
    from kineret.experiments.interpretability import embedding_geometry
    checkpoint = selected_checkpoint(UMAP_ARM)
    model, tokenizer, scaler = load_encoder(UMAP_ARM, cohort, checkpoint, 4, DEVICE)
    geometry = embedding_geometry(model, EXPERIMENT_ROOT / 'embedding_geometry')
    W.write_json(EXPERIMENT_ROOT / 'embedding_geometry' / 'checkpoint.json', W._stamp(checkpoint))
    del model, tokenizer, scaler


## Sparse-transcoder attribution to named concepts

Intended claim: **Local attribution on the trained model identifies named clinical states, trends and QA patterns associated with its predictions; selected patterns can be assessed for coherence with clinical knowledge.**

The named concepts come from input tokens; sparse latent features are not assumed to be medical concepts. Gradient?activation is local sensitivity, not causal treatment effect or an exact additive explanation of the full decision. Attribution to QA does not imply improved predictive performance.

Fit transcoders on sampled training people, validate reconstruction/sparsity on separate validation people, explain test people. Predictor weights never change; hook-forward equality is checked. Capture one layer at a time, capped at 256 evenly spaced nonpadding tokens per admission. START/END scores collapse to a named concept/value. Export signed per-admission contributions, aggregate panels, exploratory selected-concept bootstrap intervals and a clinical-review template. Clinical coherence is a human assessment; do not claim clinical validation merely because names are familiar. The intervals do not measure variation over model/transcoder fits.


In [ ]:
if RUN['xai']:
    from kineret.experiments.interpretability import sparse_xai
    xai = sparse_xai(XAI_ARM, cohort, selected_checkpoint(XAI_ARM),
        EXPERIMENT_ROOT / 'xai', device=DEVICE, n_fit=XAI_FIT_PEOPLE,
        n_validation=XAI_VALIDATION_PEOPLE, n_explain=XAI_TEST_PEOPLE,
        tokens_per_admission=XAI_TOKENS_PER_ADMISSION, n_epochs=XAI_EPOCHS)
    display(xai.sort_values('mean_contribution', ascending=False).head(20))


## Mediator abstraction stability at 96 hours

Use the actual deployed **Kineret** repository, not the MIMIC repository by accident. Direct numeric state/trend definitions are discovered from the saved repository; exclusions are exported. Perturb values by ?1%/?5% of local cutoff spacing or independently remove 5%/10% of observations, twice per condition. Re-run the real Mediator TAK implementations, including an unchanged-input control. Compare matching-label duration / union coverage and report lost/added coverage.

Sample one eligible test admission per person, then stratify by hospital and observation density; use sampling weights and stratified person bootstrap. This bounds compute and avoids repeated-admission dependence. State-threshold proximity is exported on this sample. This does not claim robustness of QA patterns, contexts, derived numeric concepts, labels or downstream predictions.


In [ ]:
if RUN['stability']:
    if MEDIATOR_REPOSITORY is None:
        raise ValueError('Set MEDIATOR_REPOSITORY to the actual deployed Kineret TAK pickle.')
    from kineret.experiments.stability import run_stability
    stability = run_stability(cohort, EXPERIMENT_ROOT / 'mediator_stability',
        MEDIATOR_ROOT, MEDIATOR_REPOSITORY, VISITS_MASTER,
        n_people=STABILITY_PEOPLE, repeats=STABILITY_REPEATS, n_resamples=BOOTSTRAP)
    display(stability)


## Deferred methods and reporting checklist

- **SAX:** implemented as a fixed SAX-style adaptation: per-concept mean/std from training observations, four-hour segment means, seven Gaussian-quantile symbols, empty segments left missing, contiguous identical observed symbols merged after discarding unfinished segments. Its vocabulary/scalers/checkpoints are separate. SAX is included in `ARMS` alongside the other models; `RUN['train_missing']` controls missing-model training for all arms. No SAX+QA arm is registered. The original LR/STraTS QA controls are retained as previously agreed.
- Multi-seed training and point-event ablations are outside this study's scope. Temporal holdout remains optional and off by default.
- Keep the frozen thesis results unchanged. Cite the new date-filtered estimates separately, with the original checkpoint training period disclosed.
- Check `status.csv`, split manifests, outcome counts and valid bootstrap draws before treating any export as complete. Best-F1 from a test threshold search is not a deployment operating point; this workflow emphasizes threshold-free discrimination, calibration and regression errors.
- Existing sigma checkpoints preserve their original preprocessing. The legacy sigma builder estimated cutoffs across the supplied cohort; new holdout sigma fits use training admissions only. Do not retrospectively describe old sigma results as strictly train-fitted preprocessing.
